# Fase 07 — Feature engineering RAW/DERIVED

**Pergunta:** que mecanismos físicos/temporais/qualidade ainda não estão representados?

**Nota (pós fase 12):** roda sobre `train` sem augmentation (decisão da fase 05 revertida).

**Nota (refatoração P5):** a fase chama `run_raw_derived_features_pipeline` (`src/scripts/generate_features.py`)
diretamente em processo (in-process), em vez de `subprocess.run` sobre o script — mesmo código, mesmos
artefatos, execução transparente e depurável dentro do notebook.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.generate_features import run_raw_derived_features_pipeline

report = run_raw_derived_features_pipeline()
report

{'n_rows': 21612,
 'n_columns': 62,
 'new_features': ['was_renovated',
  'years_since_renovation',
  'has_basement',
  'basement_ratio',
  'grade_condition_interaction',
  'log_sqft_lot',
  'bathrooms_per_bedroom',
  'sqft_living_to_lot_ratio',
  'property_cluster_distance'],
 'output_parquet': 'data/trusted/features_raw_derived.parquet'}

In [3]:
import pandas as pd

df = pd.read_parquet(report["output_parquet"])
train = df[df["split"] == "train"]

print(f"shape total: {df.shape}  (n_rows={report['n_rows']}, n_columns={report['n_columns']})")
train[report["new_features"]].describe()

shape total: (21612, 62)  (n_rows=21612, n_columns=62)


,was_renovated,years_since_renovation,has_basement,basement_ratio,grade_condition_interaction,log_sqft_lot,bathrooms_per_bedroom,sqft_living_to_lot_ratio,property_cluster_distance
count,15100.000000,15100.000000,15100.000000,15100.000000,15100.000000,15100.000000,15100.000000,15100.000000,15100.000000
mean,0.043841,-0.160728,0.390000,0.122621,26.003974,8.986683,0.647173,0.329462,2.221361
std,0.204748,5.043088,0.487766,0.169698,5.809273,0.914380,0.214064,0.271721,1.220995
min,0.000000,-1.000000,0.000000,0.000000,1.000000,6.255750,0.000000,0.001499,0.756662
25%,0.000000,-1.000000,0.000000,0.000000,21.000000,8.517393,0.500000,0.157910,1.485074
50%,0.000000,-1.000000,0.000000,0.000000,24.000000,8.922792,0.625000,0.254545,1.951381
75%,0.000000,-1.000000,1.000000,0.267879,28.000000,9.276620,0.812500,0.419527,2.662244
max,1.000000,80.000000,1.000000,0.666667,60.000000,13.887104,2.500000,4.653846,28.513087


In [4]:
corr_new = train[["price_log"] + report["new_features"]].corr()["price_log"].drop("price_log")
corr_new.sort_values(key=abs, ascending=False).to_frame("corr_price_log")

,corr_price_log
grade_condition_interaction,0.530156
property_cluster_distance,0.338222
bathrooms_per_bedroom,0.301746
has_basement,0.192679
sqft_living_to_lot_ratio,0.172481
log_sqft_lot,0.162073
basement_ratio,0.151927
was_renovated,0.124305
years_since_renovation,0.072751


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].hist(train["sqft_lot"], bins=60, color="#4C72B0", alpha=0.6)
axes[0].set_title(f"sqft_lot (skew={train['sqft_lot'].skew():.2f})")
axes[1].hist(train["log_sqft_lot"], bins=60, color="#55A868")
axes[1].set_title(f"log_sqft_lot (skew={train['log_sqft_lot'].skew():.2f})")
plt.tight_layout()
plt.savefig("reports/figures/07_raw_derived.png", dpi=110)
plt.close()
print("Figura salva em reports/figures/07_raw_derived.png")

Figura salva em reports/figures/07_raw_derived.png


## Candidatas novas — dirigidas pelos achados das fases 04/06

As 6 features acima seguem o mecanismo CONDITION/AGE (já provado fraco na EDA: `condition` r=0,039
isolado). As 3 abaixo miram no mecanismo SIZE/cobertura, com evidência direta das extensões exploratórias:

- **`bathrooms_per_bedroom`** — `bathrooms` foi a feature crua com maior correlação entre gap de
  cobertura e erro real (r=0,25) na extensão da fase 04.
- **`sqft_living_to_lot_ratio`** — `sqft_lot`/`sqft_lot15` tiveram o maior gap de distribuição
  train→test (20-21%) na mesma extensão; ataca a mesma escassez por outro ângulo do que `log_sqft_lot`
  (já rejeitado na ablation).
- **`property_cluster_distance`** — distância contínua ao centróide do `property_cluster` (fase 06),
  reusando o `KMeans`/`scaler` já fitados em `train` (`artifacts/property_cluster_model.pkl`, não fita
  nada novo aqui). A extensão da fase 06 achou o cluster raro (perfil de luxo físico) com erro médio
  4,5x pior em `test`, mas categoria instável entre partições — a versão contínua preserva o sinal sem
  herdar essa instabilidade.

Registradas em `data/processing/feature_registry.csv` como `candidate` (aguardando ablation formal da
fase 09, igual as outras 6).

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].hist(train["bathrooms_per_bedroom"].replace([float("inf")], pd.NA).dropna(), bins=60, color="#4C72B0")
axes[0].set_title("bathrooms_per_bedroom")

axes[1].hist(train["sqft_living_to_lot_ratio"], bins=60, color="#DD8452")
axes[1].set_title("sqft_living_to_lot_ratio")

axes[2].hist(train["property_cluster_distance"], bins=60, color="#55A868")
axes[2].set_title("property_cluster_distance")
plt.tight_layout()
plt.savefig("reports/figures/07_new_candidates.png", dpi=110)
plt.close()
print("Figura salva em reports/figures/07_new_candidates.png")

Figura salva em reports/figures/07_new_candidates.png


## Análise

Execução em processo confirma números idênticos aos da versão pré-reestruturação para as 6 features
originais: 21.612 linhas. Com as 3 candidatas novas (dirigidas pelas extensões das fases 04/06),
`n_columns` sobe de 59 para 62, gravadas em `data/trusted/features_raw_derived.parquet`. Em correlação
bruta com `price_log` (TRAIN, n=15.100), `grade_condition_interaction` continua a mais forte isolada
(0,530), mas logo em seguida vêm duas das 3 candidatas novas: `property_cluster_distance` (0,338) e
`bathrooms_per_bedroom` (0,302) — ambas superam `has_basement` (0,193), a segunda mais forte do grupo
original. `sqft_living_to_lot_ratio` (0,172) também supera `log_sqft_lot` (0,162), o derivado de
`sqft_lot` que já existia. `was_renovated` (0,124) e `years_since_renovation` (0,073) seguem as mais
fracas. Correlação bruta é só triagem: a decisão real de manter/descartar cada feature (as 6 originais
e as 3 novas) é da ablation formal na fase 09, não daqui — mas o sinal isolado das 3 novas já é mais
forte que a maioria das 6 antigas, o que reforça que mirar diretamente nas features que a fase 04
apontou como mais ligadas ao erro real (em vez de mecanismo CONDITION/AGE, já fraco desde a EDA) foi
uma escolha melhor dirigida.

![raw/derived](../reports/figures/07_raw_derived.png)
![candidatas novas](../reports/figures/07_new_candidates.png)

## Decisão

`data/trusted/features_raw_derived.parquet` gerado (sem augmentation), agora com 9 features candidatas
(6 originais + 3 novas dirigidas pelas fases 04/06). Todas seguem `candidate` em
`data/processing/feature_registry.csv` até a fase 09.

## Artefatos

`data/trusted/features_raw_derived.parquet`, `reports/figures/07_raw_derived.png`,
`reports/figures/07_new_candidates.png`, `data/processing/feature_registry.csv` (3 linhas novas),
`src/features/derived.py` (`add_bathrooms_per_bedroom`, `add_sqft_living_to_lot_ratio`),
`src/features/physical_typicality.py` (`add_property_cluster_distance`, novo módulo)